# Задание 2. Скалярный autograd

Реализация класса `Node`: сложение, умножение, ReLU и обратное распространение градиента по вычислительному графу. Из сторонних библиотек используется только `torch`, без `torch.autograd`, `torch.nn` и встроенного `Tensor.backward()`. Проверки написаны на стандартном `unittest`.

Материалы: [условие, задание 2](pytorch.md) и раздел «Градиенты» в [вводном ноутбуке](../notebooks/pytorch_intro.ipynb). Во вводном ноутбуке PyTorch строит граф автоматически; здесь граф и все локальные производные реализованы вручную.

**Ошибка в примере условия.** Для `a = 2`, `b = -3`, `c = 10` имеем `d = a + b*c = -28`, `e = ReLU(d) = 0`. Поскольку `d < 0`, производная ReLU равна нулю. После `e.backward()` правильные градиенты: `a.grad = b.grad = c.grad = d.grad = 0`, `e.grad = 1`. Приведённый в условии вывод этому противоречит. Если вызвать `d.backward()`, без ReLU, градиенты будут `1, 10, -3` для `a, b, c` соответственно.

## Правило цепочки и порядок обхода

Пусть `out` — результат локальной операции, а $g = \partial L / \partial out$ — уже вычисленный градиент конечного скаляра $L$.

| Операция | Передаваемый градиент |
|---|---|
| $out = x + y$ | $x.grad \mathrel{+}= g$, $y.grad \mathrel{+}= g$ |
| $out = xy$ | $x.grad \mathrel{+}= yg$, $y.grad \mathrel{+}= xg$ |
| $out = \max(0, x)$ | $x.grad \mathrel{+}= \mathbf{1}_{x>0}g$ |

В точке $x=0$ ReLU не дифференцируема; выбираем субградиент **0**. Вклады суммируются: например, в `x*x` переменная участвует в двух аргументах, и обе локальные производные должны попасть в `x.grad`.

Каждая операция сохраняет родителей и функцию `_backward`. Метод `backward()` строит топологический порядок и проходит его в обратном направлении, начиная с $\partial L/\partial L=1$. Узел обрабатывается после всех зависящих от него узлов, поэтому к этому моменту все вклады в его градиент уже собраны. Обход итеративный, без ограничения на глубину рекурсии. Время и дополнительная память — $O(V+E)$ для достижимой части графа.

**Контракт реализации:**

- Вход — действительное число Python либо нульмерный вещественный/целочисленный тензор. Данные копируются в скалярный `torch.float64` на CPU, без встроенного графа PyTorch; тензоры с `requires_grad=True` отклоняются.
- Поддерживаются числа справа и слева: `x + 2`, `2 + x`, `x * 2`, `2 * x`.
- Градиенты листьев накапливаются между вызовами `backward()`, как в учебном примере. Метод `zero_grad()` обнуляет градиент **только выбранного узла**.
- Градиенты промежуточных узлов пересчитываются для каждого прохода; их старые значения не участвуют в следующем распространении. Это позволяет повторно вызывать `backward()` на одном графе без ошибочного повторного учёта старых вкладов.
- Обновление входов требует нового прямого прохода. Поддерживаются только первые производные скалярных функций.

In [1]:
from numbers import Real
import sys
import unittest

import torch

print(f"PyTorch: {torch.__version__}; вычисления: float64, CPU")

PyTorch: 2.14.1; вычисления: float64, CPU


In [2]:
class Node:
    """Скалярный узел с собственным графом и ручным вычислением градиентов."""

    def __init__(self, data, _children=(), _op=""):
        if isinstance(data, torch.Tensor):
            if data.ndim != 0:
                raise ValueError("Node принимает только нульмерные тензоры")
            if data.requires_grad:
                raise ValueError("Встроенный autograd PyTorch не используется")
            if data.is_complex() or data.dtype == torch.bool:
                raise TypeError("Ожидается действительное число")
            self.data = data.to(device="cpu", dtype=torch.float64, copy=True)
        elif isinstance(data, Real) and not isinstance(data, bool):
            self.data = torch.tensor(float(data), dtype=torch.float64)
        else:
            raise TypeError("Ожидается действительное число или скалярный тензор")

        self.grad = torch.zeros_like(self.data)
        self._prev = tuple(_children)
        self._op = _op
        self._backward = lambda: None

    def __repr__(self):
        return f"Node(data={self.data.item():g}, grad={self.grad.item():g})"

    def __add__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        out = Node(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    __radd__ = __add__

    def __mul__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        out = Node(self.data * other.data, (self, other), "*")
        # Производные относятся к значениям именно этого прямого прохода.
        left_value = self.data.clone()
        right_value = other.data.clone()

        def _backward():
            self.grad += right_value * out.grad
            other.grad += left_value * out.grad

        out._backward = _backward
        return out

    __rmul__ = __mul__

    def relu(self):
        out = Node(self.data.clamp_min(0), (self,), "ReLU")
        local_grad = (self.data > 0).to(dtype=torch.float64)

        def _backward():
            self.grad += local_grad * out.grad

        out._backward = _backward
        return out

    def zero_grad(self):
        """Обнулить градиент этого узла, не затрагивая остальной граф."""
        self.grad.zero_()

    def backward(self):
        """Посчитать градиенты этого скаляра; накапливать их у листьев."""
        topo = []
        visited = set()
        stack = [(self, False)]
        while stack:
            node, expanded = stack.pop()
            if expanded:
                topo.append(node)
            elif node not in visited:
                visited.add(node)
                stack.append((node, True))
                stack.extend((parent, False) for parent in reversed(node._prev))

        # Старые промежуточные градиенты не должны распространяться повторно.
        for node in topo:
            if node._prev:
                node.zero_grad()
        self.grad += 1.0  # Если корень — лист, его градиент тоже накапливается.
        for node in reversed(topo):
            node._backward()

## Примеры с аналитической проверкой

Сначала воспроизведём выражение из условия с правильным результатом, затем проверим активную ветвь ReLU. В последнем примере один и тот же узел участвует в нескольких путях графа.

In [3]:
a, b, c = Node(2), Node(-3), Node(10)
d = a + b * c
e = d.relu()
e.backward()
print("Пример из условия:")
for name, node in zip("abcde", (a, b, c, d, e)):
    print(f"{name}: {node}")
assert [node.grad.item() for node in (a, b, c, d, e)] == [0, 0, 0, 0, 1]

Пример из условия:
a: Node(data=2, grad=0)
b: Node(data=-3, grad=0)
c: Node(data=10, grad=0)
d: Node(data=-28, grad=0)
e: Node(data=0, grad=1)


In [4]:
a, b, c = Node(2), Node(3), Node(10)
d = a + b * c
e = d.relu()
e.backward()
print("Активная ветвь ReLU, e = 32:")
for name, node in zip("abcde", (a, b, c, d, e)):
    print(f"{name}: {node}")
assert [node.grad.item() for node in (a, b, c, d, e)] == [1, 10, 3, 1, 1]

Активная ветвь ReLU, e = 32:
a: Node(data=2, grad=1)
b: Node(data=3, grad=10)
c: Node(data=10, grad=3)
d: Node(data=32, grad=1)
e: Node(data=32, grad=1)


In [5]:
x = Node(2)
u = x * x
loss = u * u + u  # L = x**4 + x**2; dL/dx = 4*x**3 + 2*x = 36.
loss.backward()
print(f"Первый проход: L={loss.data.item():g}, dL/du={u.grad.item():g}, x.grad={x.grad.item():g}")
assert x.grad.item() == 36
loss.backward()
print(f"Второй проход с накоплением: x.grad={x.grad.item():g}")
assert x.grad.item() == 72
x.zero_grad()
loss.backward()
print(f"После zero_grad() и нового прохода: x.grad={x.grad.item():g}")
assert x.grad.item() == 36

Первый проход: L=20, dL/du=9, x.grad=36
Второй проход с накоплением: x.grad=72
После zero_grad() и нового прохода: x.grad=36


## Тесты `unittest`

Проверяем прямые значения и аналитические производные каждой операции, ReLU в отрицательной области и в нуле, повторяющиеся аргументы, разветвления и пути разной длины, накопление и обнуление градиентов, независимые графы и цепочку из 3000 операций.

Дополнительно проверяем составную функцию

$$f(x,y,z)=\operatorname{ReLU}(xy+z)(x+y^2)+\operatorname{ReLU}(xz)$$

на 80 воспроизводимых случайных тройках. Для каждого аргумента сравниваем ручной обратный проход с центральной конечной разностью

$$\frac{\partial f}{\partial x_i}\approx\frac{f(x+\varepsilon e_i)-f(x-\varepsilon e_i)}{2\varepsilon},\qquad\varepsilon=10^{-6}.$$

Эталонная функция вычисляет только значения обычными тензорными операциями, без нашего `Node`. Точки ближе `0.15` к изломам по значениям `xy+z` или `xz` исключаются; границу ReLU проверяем отдельно согласно выбранному субградиенту. Встроенный autograd не используется даже как эталон.

In [6]:
class TestNode(unittest.TestCase):
    def assert_scalar(self, actual, expected, delta=1e-10):
        self.assertEqual(actual.ndim, 0)
        self.assertAlmostEqual(actual.item(), expected, delta=delta)

    def test_scalar_inputs(self):
        for value in (2, -1.25, torch.tensor(3), torch.tensor(0.5)):
            with self.subTest(value=value):
                node = Node(value)
                self.assert_scalar(node.data, float(value))
                self.assert_scalar(node.grad, 0)
                self.assertEqual(node.data.dtype, torch.float64)
                self.assertEqual(node.data.device.type, "cpu")

    def test_tensor_input_is_copied(self):
        source = torch.tensor(2.0)
        node = Node(source)
        source.fill_(100)
        self.assert_scalar(node.data, 2)

    def test_invalid_input(self):
        for value in ([1, 2], "2", 1j, True, torch.tensor(True), torch.tensor(1j)):
            with self.subTest(value=value), self.assertRaises(TypeError):
                Node(value)
        for value in (torch.tensor([1.0]), torch.ones(2, 2)):
            with self.subTest(shape=value.shape), self.assertRaises(ValueError):
                Node(value)

    def test_addition(self):
        x, y = Node(2), Node(-3)
        out = x + y
        out.backward()
        self.assert_scalar(out.data, -1)
        self.assert_scalar(out.grad, 1)
        self.assert_scalar(x.grad, 1)
        self.assert_scalar(y.grad, 1)

    def test_multiplication(self):
        for left, right in ((2, -3), (-2, -3), (0, 4), (2.5, 0.4)):
            with self.subTest(left=left, right=right):
                x, y = Node(left), Node(right)
                out = x * y
                out.backward()
                self.assert_scalar(out.data, left * right)
                self.assert_scalar(x.grad, right)
                self.assert_scalar(y.grad, left)

    def test_relu(self):
        for value, derivative in ((-3, 0), (0, 0), (3, 1)):
            with self.subTest(value=value):
                x = Node(value)
                out = x.relu()
                out.backward()
                self.assert_scalar(out.data, max(0, value))
                self.assert_scalar(x.grad, derivative)
                self.assert_scalar(out.grad, 1)

    def test_repeated_addition_argument(self):
        x = Node(3)
        out = x + x
        out.backward()
        self.assert_scalar(out.data, 6)
        self.assert_scalar(x.grad, 2)

    def test_repeated_multiplication_argument(self):
        for value in (-3, 0, 3):
            with self.subTest(value=value):
                x = Node(value)
                out = x * x
                out.backward()
                self.assert_scalar(out.data, value * value)
                self.assert_scalar(x.grad, 2 * value)

    def test_shared_intermediate(self):
        x = Node(2)
        u = x * x
        out = u * u + u
        out.backward()
        self.assert_scalar(out.data, 20)
        self.assert_scalar(u.grad, 9)
        self.assert_scalar(x.grad, 36)

    def test_paths_of_different_lengths(self):
        for reverse_order in (False, True):
            with self.subTest(reverse_order=reverse_order):
                x = Node(2)
                u = x * x
                out = x + u * x if reverse_order else u * x + x
                out.backward()  # x**3 + x -> 3*x**2 + 1.
                self.assert_scalar(out.data, 10)
                self.assert_scalar(x.grad, 13)

    def test_numeric_operands_and_sum(self):
        x = Node(2)
        out = sum([x + 1, 2 + x, x * 3, 4 * x])
        out.backward()
        self.assert_scalar(out.data, 21)
        self.assert_scalar(x.grad, 9)

    def test_relu_upstream_gradient(self):
        x = Node(3)
        out = -2 * (x * 4 + 1).relu()
        out.backward()
        self.assert_scalar(out.data, -26)
        self.assert_scalar(x.grad, -8)

    def test_relu_blocks_only_inactive_path(self):
        x = Node(-2)
        out = x.relu() * 5 + x * x
        out.backward()
        self.assert_scalar(out.data, 4)
        self.assert_scalar(x.grad, -4)

    def test_corrected_assignment_example(self):
        a, b, c = Node(2), Node(-3), Node(10)
        d = a + b * c
        e = d.relu()
        e.backward()
        self.assert_scalar(d.data, -28)
        self.assert_scalar(e.data, 0)
        for node in (a, b, c, d):
            self.assert_scalar(node.grad, 0)
        self.assert_scalar(e.grad, 1)
        d.backward()
        for node, expected in ((a, 1), (b, 10), (c, -3), (d, 1)):
            self.assert_scalar(node.grad, expected)

    def test_active_assignment_example(self):
        a, b, c = Node(2), Node(3), Node(10)
        out = (a + b * c).relu()
        out.backward()
        self.assert_scalar(out.data, 32)
        for node, expected in ((a, 1), (b, 10), (c, 3)):
            self.assert_scalar(node.grad, expected)

    def test_leaf_backward(self):
        x = Node(7)
        x.backward()
        self.assert_scalar(x.grad, 1)
        x.backward()
        self.assert_scalar(x.grad, 2)
        x.zero_grad()
        x.backward()
        self.assert_scalar(x.grad, 1)

    def test_repeated_backward_and_zero_grad(self):
        x = Node(2)
        u = x * x
        out = u * u + u
        out.backward()
        out.backward()
        self.assert_scalar(x.grad, 72)
        self.assert_scalar(u.grad, 9)
        self.assert_scalar(out.grad, 1)
        x.zero_grad()
        out.backward()
        self.assert_scalar(x.grad, 36)

    def test_multiple_outputs_share_graph(self):
        x = Node(2)
        u = x * x
        first, second = 2 * u, 3 * u
        first.backward()
        second.backward()
        self.assert_scalar(x.grad, 20)
        self.assert_scalar(u.grad, 3)

    def test_unconnected_node(self):
        x, unrelated = Node(2), Node(9)
        (x * 3).backward()
        self.assert_scalar(x.grad, 3)
        self.assert_scalar(unrelated.grad, 0)

    def test_long_chain(self):
        x = Node(1)
        out = x
        for _ in range(3000):
            out = out + 1
        out.backward()
        self.assert_scalar(out.data, 3001)
        self.assert_scalar(x.grad, 1)

    def test_no_native_gradient_graph(self):
        x, y = Node(2), Node(-1)
        product = x * y
        activated = (product + 4).relu()
        out = activated * x
        out.backward()
        stack, seen = [out], set()
        while stack:
            node = stack.pop()
            if node in seen:
                continue
            seen.add(node)
            for tensor in (node.data, node.grad):
                self.assertFalse(tensor.requires_grad)
                self.assertIsNone(tensor.grad_fn)
            stack.extend(node._prev)

    def test_central_finite_differences(self):
        def reference(values):
            x, y, z = values
            return (x * y + z).clamp_min(0) * (x + y * y) + (x * z).clamp_min(0)

        generator = torch.Generator().manual_seed(42)
        eps = 1e-6
        checked = 0
        max_error = 0.0
        patterns = set()
        for _ in range(1000):
            values = 4 * torch.rand(3, generator=generator, dtype=torch.float64) - 2
            x0, y0, z0 = values.tolist()
            if min(abs(x0 * y0 + z0), abs(x0 * z0)) < 0.15:
                continue
            patterns.add((x0 * y0 + z0 > 0, x0 * z0 > 0))
            x, y, z = nodes = [Node(value) for value in values]
            out = (x * y + z).relu() * (x + y * y) + (x * z).relu()
            out.backward()
            self.assert_scalar(out.data, reference(values).item())
            for index, node in enumerate(nodes):
                plus, minus = values.clone(), values.clone()
                plus[index] += eps
                minus[index] -= eps
                numerical = ((reference(plus) - reference(minus)) / (2 * eps)).item()
                max_error = max(max_error, abs(node.grad.item() - numerical))
                with self.subTest(point=checked, argument=index):
                    self.assert_scalar(node.grad, numerical, delta=1e-6)
            checked += 1
            if checked == 80:
                break
        self.assertEqual(checked, 80)
        self.assertEqual(len(patterns), 4)  # Все комбинации активности двух ReLU.
        print(f"\nКонечные разности: {checked} точек, {3 * checked} производных; "
              f"максимальная абсолютная ошибка {max_error:.3e}")

In [7]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestNode)
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
assert result.wasSuccessful(), "Есть ошибки в реализации Node"
print(f"\nУспешно пройдено тестов: {result.testsRun}")

test_active_assignment_example (__main__.TestNode.test_active_assignment_example) ... ok
test_addition (__main__.TestNode.test_addition) ... ok
test_central_finite_differences (__main__.TestNode.test_central_finite_differences) ... 
Конечные разности: 80 точек, 240 производных; максимальная абсолютная ошибка 3.078e-09
ok
test_corrected_assignment_example (__main__.TestNode.test_corrected_assignment_example) ... ok
test_invalid_input (__main__.TestNode.test_invalid_input) ... ok
test_leaf_backward (__main__.TestNode.test_leaf_backward) ... ok
test_long_chain (__main__.TestNode.test_long_chain) ... ok
test_multiple_outputs_share_graph (__main__.TestNode.test_multiple_outputs_share_graph) ... ok
test_multiplication (__main__.TestNode.test_multiplication) ... ok
test_no_native_gradient_graph (__main__.TestNode.test_no_native_gradient_graph) ... ok
test_numeric_operands_and_sum (__main__.TestNode.test_numeric_operands_and_sum) ... ok
test_paths_of_different_lengths (__main__.TestNode.test_p

## Вывод

`Node` вычисляет значения и первые производные для сложения, умножения и ReLU. Обратный топологический обход обеспечивает правильный порядок вычислений, а сложение локальных вкладов учитывает повторяющиеся аргументы и общие части графа. Проверки сравнивают результат с аналитическими производными и независимыми численными оценками. Пример из условия воспроизведён с исправленными градиентами.

Ноутбук выполняется сверху вниз на CPU. Для запуска нужен `torch`; данные, обучение модели и дополнительные сторонние библиотеки не требуются.